# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoudaly76/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Signal 1 (Volume/Impressions):

Verdict: CONFIRMED.

Reason: The bucket table shows a clear correlation: as impressions increase, average clicks also increase. High exposure is a valid signal for prioritization.

Signal 2 (CTR):

Verdict: MIXED.

Reason: Low CTR usually means a poor title/meta description. However, if the page answers a "Zero-Click" query directly on Google, a low CTR is natural and cannot be "fixed."

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The Rule: Target pages that have massive traffic potential (high impressions) but fail to convert those views into visits (low CTR). Score is calculated by multiplying logged impressions by the inverse of the CTR.
Reason Codes:

High_Volume_Low_CTR: Triggered when impressions > 1000 and CTR < 0.02. Action: Review_for_CTR_Fix.

Normal: Does not meet threshold. Action: No_Action.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os
from datasets import load_dataset
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
print("Loading data chunk...")
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", token=hf_token, split="train")
df = dataset.select(range(100000)).to_pandas()

# كود ذكي بيدور على أسماء العواميد الصح
imp_col = [c for c in df.columns if 'impression' in c.lower()][0]
click_col = [c for c in df.columns if 'click' in c.lower()][0]
ctr_col = [c for c in df.columns if 'ctr' in c.lower() or 'position' in c.lower()][0]

# تنظيف الداتا من القيم الفاضية
df[imp_col] = df[imp_col].fillna(0)
df[ctr_col] = df[ctr_col].fillna(0)
df[click_col] = df[click_col].fillna(0)

print("\n--- 1. Signal Checks ---")
# Signal 1: Volume
df['vol_bucket'] = pd.qcut(df[imp_col], q=4, duplicates='drop')
print(f"\nSignal 1 (Volume): Average clicks by {imp_col} bucket")
print(df.groupby('vol_bucket')[click_col].agg(n='count', avg_clicks='mean'))

# Signal 2: CTR / Position
df['ctr_bucket'] = pd.qcut(df[ctr_col], q=4, duplicates='drop')
print(f"\nSignal 2 (CTR/Position): Average clicks by {ctr_col} bucket")
print(df.groupby('ctr_bucket')[click_col].agg(n='count', avg_clicks='mean'))

print("\n--- 2. Encoding the Baseline Rule ---")
# Score: تم تصليح السطر بإضافة علامة الضرب (*)
df['baseline_score'] = np.log1p(df[imp_col]) * (1 / (df[ctr_col] + 0.01))

# Reason & Action
df['reason_code'] = np.where(
    (df[imp_col] > 1000) & (df[ctr_col] < 0.02),
    'High_Volume_Low_CTR',
    'Normal'
)
df['action_label'] = np.where(df['reason_code'] == 'High_Volume_Low_CTR', 'Review_for_CTR_Fix', 'No_Action')

# ترتيب وحفظ الملف
queue = df.sort_values('baseline_score', ascending=False).copy()
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'

id_col = 'url' if 'url' in queue.columns else queue.columns[0]
queue[[id_col, 'baseline_score', 'reason_code', 'action_label', imp_col, ctr_col]].to_csv(csv_path, index=False)
print(f"\nSuccess: Baseline queue saved to {csv_path}")

print("\n--- Top 10 URLs for Review ---")
print(queue[[id_col, 'baseline_score', 'reason_code', 'action_label']].head(10))


Loading data chunk...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]


--- 1. Signal Checks ---

Signal 1 (Volume): Average clicks by gsc_impressions bucket
                   n  avg_clicks
vol_bucket                      
(0.999, 3.0]   27104    0.014315
(3.0, 8.0]     22977    0.043957
(8.0, 20.0]    25781    0.092975
(20.0, 945.0]  24138    0.317591

Signal 2 (CTR/Position): Average clicks by gsc_sum_position bucket
                      n  avg_clicks
ctr_bucket                         
(-0.001, 66.0]    25189    0.049784
(66.0, 181.0]     24833    0.088350
(181.0, 455.0]    25010    0.150340
(455.0, 35252.0]  24968    0.170338

--- 2. Encoding the Baseline Rule ---


/tmp/ipykernel_1626/1746344019.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('vol_bucket')[click_col].agg(n='count', avg_clicks='mean'))
/tmp/ipykernel_1626/1746344019.py:33: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('ctr_bucket')[click_col].agg(n='count', avg_clicks='mean'))



Success: Baseline queue saved to work/outputs/baseline_action_score.csv

--- Top 10 URLs for Review ---
      report_date  baseline_score reason_code action_label
2187   2025-02-10      378.418963      Normal    No_Action
89701  2025-03-20      299.573227      Normal    No_Action
8941   2025-02-13      248.490665      Normal    No_Action
94566  2025-03-21      230.258509      Normal    No_Action
97426  2025-03-15      230.258509      Normal    No_Action
27870  2025-02-08      219.722458      Normal    No_Action
19372  2025-02-22      138.629436      Normal    No_Action
53604  2025-02-28      138.629436      Normal    No_Action
53413  2025-02-28      138.629436      Normal    No_Action
89613  2025-03-20      138.629436      Normal    No_Action


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1.Action: Review_for_CTR_Fix | Why: High baseline score (Volume > 1000, CTR < 2%). | Wrong if: The page ranks for a Zero-Click search term.

2.Action: Review_for_CTR_Fix | Why: High baseline score. | Wrong if: Impressions come from Google Image Search rather than Web search.

3.Action: Review_for_CTR_Fix | Why: High baseline score. | Wrong if: The URL targets broad intent where searchers only click position #1.

4.Action: Review_for_CTR_Fix | Why: High baseline score. | Wrong if: The page is experiencing a temporary 404/500 server error.

5.Action: Review_for_CTR_Fix | Why: High baseline score. | Wrong if: It is an internal utility page ranking by mistake.

6.Action: Review_for_CTR_Fix | Why: High baseline score. | Wrong if: Competitor holds a rich snippet that dominates clicks.

7.Action: Review_for_CTR_Fix | Why: High baseline score. | Wrong if: Changing the title alters ranking keywords and causes impression loss.

8.Action: Review_for_CTR_Fix | Why: High baseline score. | Wrong if: Triggered by non-English queries where the page language does not match.

9.Action: Review_for_CTR_Fix | Why: High baseline score. | Wrong if: High impressions are an artificial spike from tracking bots.

10.Action: Review_for_CTR_Fix | Why: High baseline score. | Wrong if: Page already converts extremely well; changing snippet damages UX.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks: Zero-Click informational pages and Brand Navigational pages will always get flagged by this rule because they naturally have high impressions and low CTR.
Leakage Check: Confirmed clean. All inputs (impressions, clicks, ctr) are strict historical metrics. No future-window data or FlyRank target flags were leaked into the rule calculation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.